# Day 2 Lab: Evidence Discipline in Practice

Bias-variance, survivorship bias, walk-forward validation, and strategy
evaluation

> **Expected Time**
>
> -   Exercise 1 (Bias-variance and bootstrap): ≈ 25 minutes
> -   Exercise 2 (Survivorship bias): ≈ 25 minutes
> -   Exercise 3 (Walk-forward strategy evaluation): ≈ 40 minutes
> -   Exercise 4 (Evidence quality assessment): ≈ 20 minutes
> -   Total: ≈ 110 minutes

<figure>
<a
href="https://colab.research.google.com/github/quinfer/minimba-notebooks/blob/main/lab02_evidence_discipline.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

## Setup (Colab-only installs)

In [ ]:
# Run this cell in Colab if a package is missing
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy import stats
    print("Setup complete.")
except Exception:
    !pip -q install numpy pandas matplotlib scipy statsmodels
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy import stats
    print("Setup complete.")

## Before You Start

This lab makes Day 2’s concepts concrete. Each exercise corresponds to a
key lesson from the lectures:

| Exercise | Day 2 concept | What you’ll do |
|-------------------|-------------------------|-----------------------------|
| 1 | Bias-variance tradeoff | See overfitting happen in real time |
| 2 | Survivorship bias | Quantify how excluding failed funds distorts returns |
| 3 | Walk-forward validation | Evaluate a simple strategy honestly |
| 4 | Evidence quality | Assess a real-world FinTech claim |

All code is provided. Your job is to **run it, change parameters,
interpret results, and write short reflections** connecting observations
to theory.

# Exercise 1 — The Bias-Variance Tradeoff (25 min)

## Why This Matters

Every model balances two errors: **bias** (too simple, misses the
pattern) and **variance** (too complex, fits the noise). In finance,
noise dominates signal, so overfitting is the default failure mode. This
exercise lets you see it happen.

## Task 1a: Fit Polynomials of Increasing Complexity

We generate data from a known function (sine wave) with noise, then fit
polynomials of different degrees. Watch what happens as complexity
increases.

In [ ]:
np.random.seed(42)
n = 30
x = np.linspace(0, 1, n)
y_true = np.sin(2 * np.pi * x)
y = y_true + np.random.normal(0, 0.3, size=n)
x_fine = np.linspace(0, 1, 200)

degrees = [1, 3, 5, 15, 25]
fig, axes = plt.subplots(1, len(degrees), figsize=(16, 3.5))

for ax, d in zip(axes, degrees):
    coef = np.polyfit(x, y, d)
    y_pred = np.polyval(coef, x_fine)
    ax.scatter(x, y, alpha=0.5, s=20, color='steelblue')
    ax.plot(x_fine, np.clip(y_pred, -2.5, 2.5), 'r-', linewidth=2)
    ax.plot(x_fine, np.sin(2 * np.pi * x_fine), 'g--', alpha=0.4, linewidth=1)
    ax.set_title(f'Degree {d}')
    ax.set_ylim(-2.5, 2.5)
    ax.grid(alpha=0.2)

plt.suptitle('Increasing Model Complexity: From Underfit to Overfit', fontsize=13)
plt.tight_layout()
plt.show()

### Questions (Write 100–150 words)

1.  At which degree does the fit look “best”? What happens at degree 25?
2.  If you only had the training data (blue dots) and no knowledge of
    the true function (green dashed), how would you choose the right
    complexity?
3.  Connect this to finance: if the “true function” is a very weak
    signal (R² ~ 1–2%), what does overfitting look like?

## Task 1b: Bootstrap Confidence Interval

The bootstrap gives us a way to quantify uncertainty without assuming a
distribution. We resample our data many times and compute the statistic
of interest on each resample.

In [ ]:
np.random.seed(42)
# Simulate 250 daily returns (1 year)
returns = np.random.normal(0.0003, 0.015, size=250)

# Bootstrap the mean
n_boot = 5000
boot_means = np.array([np.mean(np.random.choice(returns, size=len(returns), replace=True))
                        for _ in range(n_boot)])

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
sample_mean = returns.mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(boot_means, bins=50, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(sample_mean, color='red', linewidth=2, label=f'Sample mean: {sample_mean:.5f}')
ax.axvline(ci_low, color='orange', linestyle='--', linewidth=2, label=f'95% CI: [{ci_low:.5f}, {ci_high:.5f}]')
ax.axvline(ci_high, color='orange', linestyle='--', linewidth=2)
ax.axvline(0, color='black', linewidth=1, linestyle=':')
ax.set_title('Bootstrap Distribution of Mean Daily Return')
ax.set_xlabel('Mean daily return')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Sample mean: {sample_mean:.6f}")
print(f"95% Bootstrap CI: [{ci_low:.6f}, {ci_high:.6f}]")
print(f"Does the CI include zero? {'Yes — not significantly different from zero' if ci_low < 0 < ci_high else 'No — significantly different from zero'}")

### Key Insight

Even with a positive sample mean, the confidence interval often includes
zero. With daily financial data, one year of observations is frequently
**not enough** to distinguish a genuine signal from noise. This is the
signal-to-noise problem in action.

# Exercise 2 — Survivorship Bias (25 min)

## Why This Matters

When you study “the market” or “mutual funds” or “hedge funds,” you
typically analyse assets that survived to the present day. Failed funds,
delisted stocks, and bankrupt companies disappear from the data. This
systematically inflates apparent returns.

## Task 2a: Simulate 100 Funds with Failures

In [ ]:
np.random.seed(42)
n_funds = 100
n_months = 120  # 10 years
failure_threshold = -0.30  # Fund closes after 30% cumulative drawdown

# All funds have IDENTICAL expected returns (no skill)
all_returns = np.random.normal(0.005, 0.04, size=(n_months, n_funds))
cumulative = np.cumprod(1 + all_returns, axis=0)

# Track survival
alive = np.ones(n_funds, dtype=bool)
death_month = np.full(n_funds, n_months)
for f in range(n_funds):
    for t in range(1, n_months):
        if cumulative[t, f] < (1 + failure_threshold) * cumulative[0, f]:
            alive[f] = False
            death_month[f] = t
            cumulative[t+1:, f] = np.nan
            all_returns[t+1:, f] = np.nan
            break

n_survivors = alive.sum()
n_failed = n_funds - n_survivors

# Average returns: all funds vs survivors only
all_mean = np.nanmean(all_returns, axis=1)
survivor_mask = all_returns.copy()
survivor_mask[:, ~alive] = np.nan
survivor_mean = np.nanmean(survivor_mask, axis=1)

cum_all = np.cumprod(1 + all_mean)
cum_survivor = np.cumprod(1 + np.nan_to_num(survivor_mean, nan=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Individual fund paths
for f in range(n_funds):
    colour = 'coral' if not alive[f] else 'steelblue'
    alpha = 0.15
    axes[0].plot(cumulative[:, f], color=colour, alpha=alpha, linewidth=0.5)
axes[0].set_title(f'{n_funds} Funds: {n_survivors} Survived (blue), {n_failed} Failed (red)')
axes[0].set_ylabel('Cumulative Growth (£1)')
axes[0].set_xlabel('Month')
axes[0].grid(alpha=0.3)

# Panel 2: Average comparison
axes[1].plot(cum_all, linewidth=2.5, label=f'All {n_funds} funds', color='steelblue')
axes[1].plot(cum_survivor, linewidth=2.5, label=f'Survivors only ({n_survivors})', color='coral')
axes[1].set_title('Survivorship Bias: Survivors Look Better')
axes[1].set_ylabel('Cumulative Growth (£1)')
axes[1].set_xlabel('Month')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

bias = (cum_survivor[-1] - cum_all[-1]) / cum_all[-1] * 100
print(f"Survivors: {n_survivors} / {n_funds}")
print(f"Final value (all funds): £{cum_all[-1]:.2f}")
print(f"Final value (survivors): £{cum_survivor[-1]:.2f}")
print(f"Survivorship bias: {bias:.1f}% overstated")

### Questions (Write 150–200 words)

1.  All 100 funds have **identical** expected returns (no skill). Yet
    the survivors appear to outperform. Why?
2.  A financial advisor shows you that “the average UK equity fund
    returned 8% annually over the last decade.” What question should you
    ask?
3.  How would this bias affect your evaluation of a FinTech lending
    platform that reports its “historical default rate” using only
    currently active loans?

## Task 2b: Change the Parameters

Try changing these parameters and observe the effect on survivorship
bias:

-   `failure_threshold = -0.20` (more sensitive trigger — more funds
    fail)
-   `failure_threshold = -0.50` (more robust — fewer fail)
-   `n_months = 60` (5 years instead of 10)
-   `np.random.seed(123)` (different random draw)

Which parameter has the biggest effect on bias magnitude?

# Exercise 3 — Walk-Forward Strategy Evaluation (40 min)

## Why This Matters

This is the core practical exercise. You will evaluate a simple
moving-average crossover strategy using walk-forward validation and
honest baselines — the same discipline that separates genuine evidence
from backtest mining.

## Task 3a: Load Data and Implement Strategy

In [ ]:
np.random.seed(42)
# Simulate daily returns for a stock (or load Bloomberg data if available)
n_days = 1260  # ~5 years
dates = pd.date_range('2019-01-01', periods=n_days, freq='B')
daily_returns = pd.Series(np.random.normal(0.0003, 0.015, size=n_days), index=dates, name='returns')

# Try loading Bloomberg data if available
try:
    df = pd.read_parquet('../../data/bloomberg_database/bloomberg_database.parquet')
    spy = df[df['ticker'] == 'SPY'].set_index('date')['PX_LAST'].sort_index()
    daily_returns = spy.pct_change().dropna()
    daily_returns.name = 'returns'
    print(f"Loaded Bloomberg SPY data: {len(daily_returns)} observations")
except Exception:
    print(f"Using simulated data: {len(daily_returns)} observations")

# Compute moving averages
prices = (1 + daily_returns).cumprod()
ma_short = prices.rolling(20).mean()
ma_long = prices.rolling(60).mean()

# Generate signal: 1 when short MA > long MA, 0 otherwise
signal = (ma_short > ma_long).astype(int).shift(1).dropna()  # shift(1) = no look-ahead

# Strategy returns
strategy_returns = signal * daily_returns.loc[signal.index]
buyhold_returns = daily_returns.loc[signal.index]

# Cumulative comparison
cum_strategy = (1 + strategy_returns).cumprod()
cum_buyhold = (1 + buyhold_returns).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum_buyhold, label='Buy & Hold (baseline)', linewidth=2, color='steelblue')
ax.plot(cum_strategy, label='MA Crossover (20/60)', linewidth=2, color='coral')
ax.set_title('Strategy vs Buy-and-Hold: Full Sample (Not Walk-Forward Yet)')
ax.set_ylabel('Cumulative Growth (£1)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Performance metrics
sharpe_strategy = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
sharpe_buyhold = buyhold_returns.mean() / buyhold_returns.std() * np.sqrt(252)
print(f"Strategy Sharpe (full sample): {sharpe_strategy:.2f}")
print(f"Buy & Hold Sharpe (full sample): {sharpe_buyhold:.2f}")

> **Warning**
>
> This is a **full-sample** backtest. It does NOT use walk-forward
> validation. The results may be misleading. We fix this next.

## Task 3b: Walk-Forward Evaluation

Now let’s evaluate the strategy properly using expanding-window
walk-forward validation.

In [ ]:
# Walk-forward parameters
min_train = 252  # Minimum 1 year of training data
test_window = 63  # Test on 3 months at a time

n = len(daily_returns)
fold_results = []
all_wf_strategy = []
all_wf_buyhold = []

start = min_train
while start + test_window <= n:
    # Training data: everything up to 'start'
    train_returns = daily_returns.iloc[:start]
    train_prices = (1 + train_returns).cumprod()
    
    # Test data: next 'test_window' days
    test_slice = slice(start, start + test_window)
    test_returns = daily_returns.iloc[test_slice]
    test_prices = (1 + daily_returns.iloc[:start + test_window]).cumprod()
    
    # Compute MAs using only data up to and including current day
    ma_s = test_prices.rolling(20).mean()
    ma_l = test_prices.rolling(60).mean()
    sig = (ma_s > ma_l).astype(int).shift(1)
    
    # Strategy returns in test window
    test_sig = sig.iloc[test_slice]
    strat_ret = test_sig * test_returns
    
    sharpe_wf = strat_ret.mean() / strat_ret.std() * np.sqrt(252) if strat_ret.std() > 0 else 0
    sharpe_bh = test_returns.mean() / test_returns.std() * np.sqrt(252) if test_returns.std() > 0 else 0
    
    fold_results.append({
        'fold': len(fold_results) + 1,
        'test_start': daily_returns.index[start],
        'sharpe_strategy': sharpe_wf,
        'sharpe_buyhold': sharpe_bh,
        'strategy_beats_bh': sharpe_wf > sharpe_bh,
    })
    
    all_wf_strategy.extend(strat_ret.tolist())
    all_wf_buyhold.extend(test_returns.tolist())
    
    start += test_window

results_df = pd.DataFrame(fold_results)

# Overall walk-forward performance
wf_strat = np.array(all_wf_strategy)
wf_bh = np.array(all_wf_buyhold)
wf_sharpe_strat = np.nanmean(wf_strat) / np.nanstd(wf_strat) * np.sqrt(252)
wf_sharpe_bh = np.nanmean(wf_bh) / np.nanstd(wf_bh) * np.sqrt(252)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Sharpe ratio by fold
axes[0].bar(results_df['fold'] - 0.15, results_df['sharpe_strategy'], width=0.3,
            label='Strategy', color='coral', alpha=0.8)
axes[0].bar(results_df['fold'] + 0.15, results_df['sharpe_buyhold'], width=0.3,
            label='Buy & Hold', color='steelblue', alpha=0.8)
axes[0].set_xlabel('Walk-Forward Fold')
axes[0].set_ylabel('Annualised Sharpe Ratio')
axes[0].set_title('Walk-Forward Sharpe by Fold')
axes[0].legend()
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].grid(alpha=0.3)

# Panel 2: Cumulative walk-forward P&L
axes[1].plot(np.cumprod(1 + np.array(all_wf_buyhold)), label='Buy & Hold', color='steelblue', linewidth=2)
axes[1].plot(np.cumprod(1 + np.array(all_wf_strategy)), label='Strategy', color='coral', linewidth=2)
axes[1].set_title('Cumulative Walk-Forward Performance')
axes[1].set_ylabel('Growth of £1')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

win_rate = results_df['strategy_beats_bh'].mean()
print(f"\nWalk-forward results:")
print(f"  Strategy Sharpe (aggregate): {wf_sharpe_strat:.2f}")
print(f"  Buy & Hold Sharpe (aggregate): {wf_sharpe_bh:.2f}")
print(f"  Strategy beats B&H in {win_rate:.0%} of folds")
print(f"  Number of folds: {len(results_df)}")
print(f"\nFull-sample Sharpe was {sharpe_strategy:.2f} — walk-forward is {wf_sharpe_strat:.2f}")
print(f"This {'supports' if wf_sharpe_strat > wf_sharpe_bh else 'does not support'} the strategy.")

### Questions (Write 200–300 words)

1.  Compare the full-sample Sharpe ratio to the walk-forward Sharpe
    ratio. Why are they different?
2.  Does the strategy consistently beat buy-and-hold across folds, or is
    performance concentrated in specific periods?
3.  Would you recommend deploying this strategy with real money? What
    additional information would you need?
4.  You tested one parameter combination (20/60 days). If you also
    tested 10/30, 20/50, 30/90, and 50/200, how would this affect your
    confidence in the result? (Hint: multiple testing.)

# Exercise 4 — Evidence Quality Assessment (20 min)

## No Code Required

Choose **one** of the following real-world claims and write a 200–300
word assessment using the frameworks from today:

**Claim A:** “Our robo-advisor has outperformed the market by 2%
annually over the last 5 years.”

**Claim B:** “Our AI credit scoring model reduces default rates by 25%
compared to traditional methods.”

**Claim C:** “Our algorithmic trading system has achieved a Sharpe ratio
of 2.5 since launch.”

**For your chosen claim, address:**

1.  **Pitfall check:** Which of the five pitfalls might apply?
    (Significance vs importance, p-hacking, publication bias, etc.)
2.  **Bias check:** Could survivorship bias, look-ahead bias, or
    selection bias explain the result?
3.  **Baseline check:** What is the relevant baseline? Does the claim
    beat it meaningfully?
4.  **Overfitting check:** Where on the overfitting spectrum does this
    claim sit?
5.  **What would you ask?** List 3 specific questions you would ask the
    vendor.

> **Connection to Assessment**
>
> This exercise is direct preparation for the final briefing note. The
> assessment asks you to evaluate a real FinTech or AI claim using
> exactly these frameworks.

# Synthesis

## Connecting the Exercises

| Exercise | Key lesson | Day 2 framework |
|--------------------|----------------------|-------------------------------|
| 1 (Bias-variance) | Complexity ≠ accuracy; noise dominates in finance | Overfitting is the default |
| 2 (Survivorship) | Missing data silently distorts results | Failure modes are invisible |
| 3 (Walk-forward) | Full-sample ≠ out-of-sample; baselines matter | Honest evaluation requires discipline |
| 4 (Assessment) | Claims need scrutiny using structured frameworks | Evidence quality is a skill |

## Final Reflection (150–200 words)

Considering all four exercises, write a short reflection:

1.  Before today, how would you have evaluated a quantitative claim from
    a FinTech vendor? What would you do differently now?
2.  Which failure mode (overfitting, survivorship, look-ahead, multiple
    testing) do you think is most commonly exploited in financial
    marketing? Why?
3.  What is the single most important question to ask when presented
    with a backtest result?

## References